# Module 302 — Label Flipping Study

HTB Academy, AI Data Attacks, Sections 3-4. This notebook follows the code introduced so far: create a small synthetic binary-classification dataset, split it, inspect the clean training data, fit a clean logistic-regression baseline, and evaluate it on held-out examples. The poisoning operation belongs in the later course step that introduces it.

The examples represent sentiment classes with two made-up numeric features. They are not actual review text, email, or production data. The notebook uses 1,000 rows so it runs quickly on CPU and does not load external files or contact a service.

## Read the notation

HTB describes an example as `x_i = (x_i1, x_i2)`. Say it aloud: **"x sub i equals the pair x sub i one, x sub i two."** `i` is the example's row number; the two values are its two numeric features. `y_i`, read **"y sub i,"** is the class label for that example. Label flipping changes `y_i` and leaves `x_i` alone. For binary labels only, a teaching shorthand is `y_i' = 1 - y_i` ("y sub i prime equals one minus y sub i"): zero becomes one, and one becomes zero. This formula is explanatory and is not yet applied by the code below.

## 1. Import the small set of tools

NumPy generates and summarizes arrays. Matplotlib draws the scatter plot. `make_blobs` creates clustered toy data, and `train_test_split` separates examples for later evaluation. The HTB cell imports additional reporting and Seaborn tools for the broader exercise; this section's displayed baseline does not need them, so this study notebook keeps the imports focused.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Fixed randomness makes the same toy dataset appear each time.
SEED = 1337
rng = np.random.default_rng(SEED)

# Reuse the lesson's dark background and clear class colors.
HTB_GREEN = '#9fef00'
CLASS_COLORS = ['#0086ff', '#ffaf00']  # class 0, class 1

## 2. Generate the synthetic examples

The mathematical shape is a feature matrix `X` with one row per example and two columns, plus a label vector `y` with one class per row. `X.shape` should therefore be `(number of examples, 2)`, while `y.shape` should be `(number of examples,)`. The centers `(0, 5)` and `(5, 0)` make two visually distinct groups. `cluster_std=1.25` controls how spread out each group is; a larger value creates more overlap.

In [ ]:
N_SAMPLES = 1_000

# Each row in X is one two-feature example x_i. Each entry in y is y_i.
X, y = make_blobs(
    n_samples=N_SAMPLES,
    centers=[(0, 5), (5, 0)],
    n_features=2,
    cluster_std=1.25,
    random_state=SEED,
)

print(f'Feature matrix X shape: {X.shape}')
print(f'Label vector y shape:   {y.shape}')
print(f'Class labels present:   {np.unique(y)}')
print(f'First example x_0:      {X[0]}')
print(f'Its label y_0:          {y[0]}')

## 3. Make a training split and an untouched test split

A 30% test split means about 300 of the 1,000 rows are held out; the remaining about 700 are for training. The fixed `random_state` makes the split reproducible. We do not alter either set's labels in this baseline cell. Later, when demonstrating poisoning, only a copy of the training labels should be changed. Keeping test labels correct lets us measure whether the model learned worse patterns from the corrupted training answers.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
)

print(f'Training examples: {len(y_train)}')
print(f'Test examples:     {len(y_test)}')
print(f'Training class counts: {np.bincount(y_train)}')
print(f'Test class counts:     {np.bincount(y_test)}')

## 4. Plot the clean training data

Each dot is one training example. Its horizontal and vertical locations are the two features in `x_i`; its color comes from `y_i`. The plot shows the starting point before any label changes. Separate clusters make this classroom example easy to see, but real data may overlap and have many more dimensions.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#141d2b')
ax.set_facecolor('#141d2b')

# X_train[:, 0] selects feature 1 for all training rows;
# X_train[:, 1] selects feature 2. The color is the unchanged label.
for class_id, color in enumerate(CLASS_COLORS):
    rows = y_train == class_id
    ax.scatter(
        X_train[rows, 0],
        X_train[rows, 1],
        color=color,
        edgecolors='#141d2b',
        alpha=0.8,
        label=f'Class {class_id}',
    )

ax.set_title('Clean training data before label flipping', color=HTB_GREEN)
ax.set_xlabel('Synthetic feature 1', color='white')
ax.set_ylabel('Synthetic feature 2', color='white')
ax.tick_params(colors='#a4b1cd')
ax.grid(alpha=0.2, color='#a4b1cd')
ax.legend()
plt.show()

## 5. Fit and evaluate the clean baseline

For example i, the model score is z_i = w^T x_i + b. Say it aloud: "z sub i equals w transpose times x sub i, plus b." The weights multiply the two feature values and the bias shifts the score. The sigmoid p_i = 1 / (1 + e^(-z_i)) converts this score to a probability of class 1. Scikit-learn learns its weights from clean training pairs only. We measure accuracy on the held-out test set, which was not used to fit the model.

In [ ]:
# Fit only on the original, clean training examples and labels.
baseline_model = LogisticRegression(random_state=SEED)
baseline_model.fit(X_train, y_train)

# Compare predictions with untouched, correct test labels.
y_pred_baseline = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print(f'Baseline model test accuracy: {baseline_accuracy:.4f}')

# HTB reports 0.9933 for this fixed, well-separated toy dataset.
# Small library or solver-version differences can affect the last decimal.

## 6. Visualize the model decision boundary

The classifier predicts class 1 when p >= 0.5. Sigmoid is 0.5 when z = 0, so the boundary is w^T x + b = 0. With two features this is a straight line. To draw it, we predict classes at nearby feature coordinates and shade each region. This visualizes the fitted model; it does not train it again. We use grid spacing 0.05 instead of HTB 0.02 to reduce temporary points while keeping the plot clear.

In [ ]:
def plot_decision_boundary(model, X_plot, y_plot, title, mesh_step=0.05):
    """Draw predicted class regions and the two-feature examples."""
    # Add padding so edge examples are not clipped.
    x_min, x_max = X_plot[:, 0].min() - 1, X_plot[:, 0].max() + 1
    y_min, y_max = X_plot[:, 1].min() - 1, X_plot[:, 1].max() + 1

    # Build a lightweight grid of feature coordinates for prediction.
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, mesh_step),
        np.arange(y_min, y_max, mesh_step),
    )
    grid_points = np.column_stack((xx.ravel(), yy.ravel()))
    grid_predictions = model.predict(grid_points).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(9, 6))
    fig.patch.set_facecolor('#141d2b')
    ax.set_facecolor('#141d2b')
    ax.contourf(
        xx, yy, grid_predictions,
        levels=[-0.5, 0.5, 1.5], colors=CLASS_COLORS, alpha=0.3,
    )

    # Overlay the original examples, colored by their clean labels.
    for class_id, color in enumerate(CLASS_COLORS):
        rows = y_plot == class_id
        ax.scatter(
            X_plot[rows, 0], X_plot[rows, 1],
            color=color, edgecolors='#141d2b', alpha=0.8,
            label=f'Class {class_id}',
        )

    ax.set_title(title, color=HTB_GREEN)
    ax.set_xlabel('Synthetic feature 1', color='white')
    ax.set_ylabel('Synthetic feature 2', color='white')
    ax.tick_params(colors='#a4b1cd')
    ax.grid(alpha=0.2, color='#a4b1cd')
    ax.legend()
    plt.show()

# Visualize the model trained above; no additional fitting happens here.
plot_decision_boundary(
    baseline_model,
    X_train,
    y_train,
    title=f'Clean baseline decision regions (test accuracy {baseline_accuracy:.4f})',
)

## Check your understanding

1. In x_i = (x_i1, x_i2), which part is the label? (Neither; the label is y_i.)
2. In label flipping, what changes: the feature row, the label, or both? (The label changes; features stay fixed.)
3. Why keep test labels clean? (So evaluation measures the model against correct answers.)
4. In z = w^T x + b, what do the weights and bias do? (They form and shift the linear score.)
5. What does sigmoid do? (It maps a score to a probability between zero and one.)
6. Why establish a clean baseline? (So later model performance can be compared against a trusted reference.)
7. Does this plot and its high accuracy demonstrate a real sentiment system? (No. It uses artificial two-dimensional data.)

Scope note: this notebook currently covers Sections 3-4: clean synthetic data and a clean logistic-regression baseline. It does not implement the poisoning step or later sections early.